In [1]:
!pip install -U transformers accelerate sentence-transformers chromadb langchain langchain-community

  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached opentelemetry_api-1.40.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.40.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-win_amd64.whl.metadata (10 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached orjson-3.11.7-cp312-cp312-win_amd64.whl.metadata (43 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached sqlalchemy-2.0.48-cp312-cp312-win_amd64.whl.metadata (9.8 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached greenlet-3.3.2-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.

In [4]:
!pip install langchain-text-splitters

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [6]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [7]:
loader = TextLoader("document.txt")  
documents = loader.load()

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

texts = text_splitter.split_documents(documents)

print("Number of chunks:", len(texts))

Number of chunks: 3


In [8]:
embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"
)

C:\Users\Piyush\AppData\Local\Temp\ipykernel_13672\2938260996.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Piyush\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Piyush\.cache\huggingface\hub\models--BAAI--bge-small-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
persist_directory = "db"

vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embedding,
    persist_directory=persist_directory
)

retriever = vectordb.as_retriever(search_kwargs={"k": 3})

In [10]:
model_name = "Qwen/Qwen2-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

print("Model loaded on:", model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

C:\Users\Piyush\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Piyush\.cache\huggingface\hub\models--Qwen--Qwen2-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded on: cuda:0


In [11]:
def generate_answer(prompt, max_new_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [12]:
prompt_template = """You are a strict QA system.

Answer ONLY from the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

In [15]:
def rag_query(query):
    docs = retriever.invoke(query)

    print("\n--- Retrieved Docs ---")
    for i, doc in enumerate(docs):
        print(f"\nDoc {i}:\n{doc.page_content[:300]}...")

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = PROMPT.format(context=context, question=query)

    print("\n--- Final Prompt ---\n")
    print(prompt[:1000])  # avoid huge prints

    response = generate_answer(prompt)

    return response

In [16]:
query = "What is the document about?"

response = rag_query(query)

print("\n--- Answer ---\n")
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Retrieved Docs ---

Doc 0:
In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms with language models....

Doc 1:
Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can generate new content such as text, images, or code. These models are typically based on transformer architectures and are trained on large datasets.

One major limitation of generative models is hallucina...

Doc 2:
Embeddings are numerical vector representations of text that capture semantic meaning. Similar texts have embeddings that are close in vector space.

Vector databases such as Chroma store embeddings and allow efficient similarity search using metrics like cosine similarity.

Chunking is an important...

--- Final Prompt ---

You are a strict QA system.

Answer ONLY from the provided context.
If the answer is not in the context, say "I don't know."

Context:
In summary, RAG improves the reliability of generative AI sys

In [17]:
query = "What is RAG?"

docs = vectordb.similarity_search(query, k=3)

for i, doc in enumerate(docs):
    print(f"\n--- Result {i} ---\n")
    print(doc.page_content)


--- Result 0 ---

Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can generate new content such as text, images, or code. These models are typically based on transformer architectures and are trained on large datasets.

One major limitation of generative models is hallucination. Hallucination occurs when a model generates information that is not grounded in reality or is factually incorrect.

Retrieval-Augmented Generation (RAG) is a technique used to reduce hallucination. Instead of relying only on the model's internal knowledge, RAG retrieves relevant documents from an external database and provides them as context to the model.

A typical RAG pipeline consists of the following steps:
1. Document ingestion
2. Text chunking
3. Embedding generation
4. Storage in a vector database
5. Query embedding
6. Similarity search
7. Context injection into the prompt
8. Response generation using an LLM

--- Result 1 ---

In summary, RAG improves the rel

In [18]:
query = "What is RAG?"

docs = vectordb.similarity_search(query, k=3)

for i, doc in enumerate(docs):
    print(f"\n--- Result {i} ---\n")
    print(doc.page_content)


--- Result 0 ---

Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can generate new content such as text, images, or code. These models are typically based on transformer architectures and are trained on large datasets.

One major limitation of generative models is hallucination. Hallucination occurs when a model generates information that is not grounded in reality or is factually incorrect.

Retrieval-Augmented Generation (RAG) is a technique used to reduce hallucination. Instead of relying only on the model's internal knowledge, RAG retrieves relevant documents from an external database and provides them as context to the model.

A typical RAG pipeline consists of the following steps:
1. Document ingestion
2. Text chunking
3. Embedding generation
4. Storage in a vector database
5. Query embedding
6. Similarity search
7. Context injection into the prompt
8. Response generation using an LLM

--- Result 1 ---

In summary, RAG improves the rel

In [20]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

docs = retriever.invoke(query)

for doc in docs:
    print(doc.page_content[:200])

Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can generate new content such as text, images, or code. These models are typically based on transformer arch
In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms with language models.
Embeddings are numerical vector representations of text that capture semantic meaning. Similar texts have embeddings that are close in vector space.

Vector databases such as Chroma store embeddings a


In [21]:
for k in [1, 3, 5]:
    print(f"\n=== k = {k} ===")
    docs = vectordb.similarity_search(query, k=k)
    for doc in docs:
        print("-", doc.page_content[:100])


=== k = 1 ===
- Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can gene

=== k = 3 ===
- Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can gene
- In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms 
- Embeddings are numerical vector representations of text that capture semantic meaning. Similar texts

=== k = 5 ===
- Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can gene
- In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms 
- Embeddings are numerical vector representations of text that capture semantic meaning. Similar texts


In [22]:
queries = [
    "What is hallucination?",
    "Explain embeddings",
    "Who invented RAG?"  # not in doc
]

for q in queries:
    print(f"\n=== Query: {q} ===")
    docs = vectordb.similarity_search(q, k=2)
    for doc in docs:
        print(doc.page_content[:150])


=== Query: What is hallucination? ===
Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can generate new content such as text, images, or code. Th
In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms with language models.

=== Query: Explain embeddings ===
Embeddings are numerical vector representations of text that capture semantic meaning. Similar texts have embeddings that are close in vector space.


In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms with language models.

=== Query: Who invented RAG? ===
In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms with language models.
Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can generate new content such as text, images, or code. Th


In [23]:
collection = vectordb._collection

print("Total docs:", collection.count())

Total docs: 3


In [24]:
data = collection.get(include=["documents", "embeddings"])

print(data.keys())
print(len(data["documents"]))

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
3


In [25]:
print(len(data["embeddings"][0]))  # vector dimension
print(data["documents"][0][:200])

384
Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can generate new content such as text, images, or code. These models are typically based on transformer arch


In [26]:
new_texts = ["RAG is widely used in production AI systems."]

vectordb.add_texts(new_texts)

print("New count:", vectordb._collection.count())

New count: 4


In [27]:
ids = vectordb._collection.get()["ids"]

vectordb._collection.delete(ids=[ids[0]])

print("After delete:", vectordb._collection.count())

After delete: 3


In [28]:
docs = vectordb.similarity_search("RAG systems", k=2)

for doc in docs:
    print(doc.page_content)

RAG is widely used in production AI systems.
In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms with language models.


In [29]:
embedding2 = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb2 = Chroma.from_documents(
    documents=texts,
    embedding=embedding2
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Piyush\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Piyush\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
docs1 = vectordb.similarity_search("RAG pipeline", k=2)
docs2 = vectordb2.similarity_search("RAG pipeline", k=2)

print("\n--- BGE ---")
for d in docs1:
    print(d.page_content[:100])

print("\n--- MiniLM ---")
for d in docs2:
    print(d.page_content[:100])


--- BGE ---
RAG is widely used in production AI systems.
In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms 

--- MiniLM ---
Generative AI and Retrieval-Augmented Generation (RAG)

Generative AI refers to models that can gene
In summary, RAG improves the reliability of generative AI systems by combining retrieval mechanisms 
